# Retrieval Accuracy Evaluation

Evaluate the accuracy of our hybrid (Semantic + BM25 + RRF + Cross-Encoder) retrieval pipeline.

**Metrics:**
- **Hit@K** — Is the correct document in the top-K results?
- **MRR** (Mean Reciprocal Rank) — At what rank does the correct document appear?
- **Precision@K** — What fraction of top-K results are relevant?

In [ ]:
import sys
import os

# Add ai_engine to path for local service imports
sys.path.append(os.path.abspath('../ai_engine'))

from services.vector_store import VectorStore
from services.bm25_index import BM25Index
from services.retrieval_service import RetrievalService
from services.reranker import Reranker

print("✅ Successfully imported search services.")

In [ ]:
# ── Initialize Services ──

vector_store = VectorStore()
bm25_index = BM25Index()
reranker = Reranker()
retrieval = RetrievalService(vector_store, bm25_index, reranker)

print(f"✅ Services initialized.")
print(f"   Qdrant URL: {vector_store.url}")
print(f"   Embedding dim: {vector_store.vector_size}")
print(f"   Cross-encoder: {reranker.model_name}")

In [ ]:
# ── Evaluation Dataset ──
# Each entry has a query, the collection to search, and the expected chunk IDs
# that should appear in the top results.

COLLECTION = "rfp_chunks"  # Change to your actual collection name

evaluation_set = [
    {
        "query": "What is the required system uptime and availability SLA?",
        "expected_ids": ["req_availability_001"],
        "category": "Technical"
    },
    {
        "query": "Is ISO 27001 certification mandatory for the vendor?",
        "expected_ids": ["req_security_002"],
        "category": "Security"
    },
    {
        "query": "What are the performance requirements for API response times?",
        "expected_ids": ["req_perf_003"],
        "category": "Performance"
    },
    {
        "query": "Describe the project management methodology and reporting cadence",
        "expected_ids": ["req_pm_004"],
        "category": "Management"
    },
    {
        "query": "What data encryption standards are required at rest and in transit?",
        "expected_ids": ["req_encryption_005"],
        "category": "Security"
    }
]

print(f"📋 Evaluation set: {len(evaluation_set)} queries across categories.")

In [ ]:
# ── Evaluation Loop ──

K_VALUES = [1, 3, 5, 10]

def compute_metrics(evaluation_set, retrieval_service, collection, k_values):
    """Run queries, compare results against ground truth, compute Hit@K and MRR."""
    results_log = []
    
    for entry in evaluation_set:
        query = entry["query"]
        expected = set(entry["expected_ids"])
        
        # Run hybrid retrieval
        retrieved = retrieval_service.retrieve(
            query, collection_name=collection, top_n=max(k_values)
        )
        retrieved_ids = [doc.get("id", "") for doc in retrieved]
        
        # Find the rank of the first relevant document (1-indexed)
        first_relevant_rank = None
        for rank, doc_id in enumerate(retrieved_ids, 1):
            if doc_id in expected:
                first_relevant_rank = rank
                break
        
        reciprocal_rank = (1.0 / first_relevant_rank) if first_relevant_rank else 0.0
        
        # Hit@K for each K value
        hits = {}
        precisions = {}
        for k in k_values:
            top_k_ids = set(retrieved_ids[:k])
            hits[f"hit@{k}"] = 1 if expected & top_k_ids else 0
            precisions[f"precision@{k}"] = len(expected & top_k_ids) / k
        
        results_log.append({
            "query": query[:60],
            "category": entry.get("category", "-"),
            "reciprocal_rank": reciprocal_rank,
            **hits,
            **precisions,
            "retrieved_count": len(retrieved),
            "top_score": retrieved[0].get("rerank_score", 0) if retrieved else 0,
        })
    
    return results_log

results = compute_metrics(evaluation_set, retrieval, COLLECTION, K_VALUES)
print(f"✅ Evaluation complete for {len(results)} queries.")

In [ ]:
# ── Aggregate Metrics ──

import json

n = len(results)
if n > 0:
    mrr = sum(r["reciprocal_rank"] for r in results) / n
    
    print("═" * 60)
    print(f"  RETRIEVAL ACCURACY REPORT ({n} queries)")
    print("═" * 60)
    print(f"  MRR (Mean Reciprocal Rank): {mrr:.4f}")
    
    for k in K_VALUES:
        hit_rate = sum(r[f"hit@{k}"] for r in results) / n
        avg_precision = sum(r[f"precision@{k}"] for r in results) / n
        print(f"  Hit@{k}: {hit_rate:.2%}  |  Precision@{k}: {avg_precision:.4f}")
    
    print("═" * 60)
    print("\n📊 Per-query breakdown:")
    for r in results:
        print(f"  [{r['category']:12s}] RR={r['reciprocal_rank']:.2f}  "
              f"H@5={'✅' if r.get('hit@5') else '❌'}  "
              f"Score={r['top_score']:.4f}  "
              f"'{r['query']}...'")
else:
    print("⚠️  No results to aggregate. Check collection name and data.")

In [ ]:
# ── Semantic-Only Baseline Comparison ──
# Run the same queries using ONLY vector search (no BM25, no reranking)

baseline_results = []
for entry in evaluation_set:
    query = entry["query"]
    expected = set(entry["expected_ids"])
    
    # Vector-only search
    vec_results = vector_store.query(COLLECTION, query, n_results=10)
    vec_ids = [doc.get("id", "") for doc in vec_results]
    
    rr = 0.0
    for rank, did in enumerate(vec_ids, 1):
        if did in expected:
            rr = 1.0 / rank
            break
    
    hit5 = 1 if expected & set(vec_ids[:5]) else 0
    baseline_results.append({"rr": rr, "hit5": hit5})

if baseline_results:
    baseline_mrr = sum(r["rr"] for r in baseline_results) / len(baseline_results)
    baseline_hit5 = sum(r["hit5"] for r in baseline_results) / len(baseline_results)
    
    print("\n═" * 60)
    print("  COMPARISON: Hybrid vs Semantic-Only")
    print("═" * 60)
    print(f"  {'Metric':<20} {'Hybrid':>10} {'Semantic':>10} {'Delta':>10}")
    print(f"  {'MRR':<20} {mrr:>10.4f} {baseline_mrr:>10.4f} {mrr - baseline_mrr:>+10.4f}")
    hybrid_hit5 = sum(r.get('hit@5', 0) for r in results) / n
    print(f"  {'Hit@5':<20} {hybrid_hit5:>10.2%} {baseline_hit5:>10.2%} {hybrid_hit5 - baseline_hit5:>+10.2%}")
    print("═" * 60)